# Week 5 — Agents, tools, MCP, state, and approval

Compare in-process, prompt, Hosted, and self-hosted agents. Hosting and protocol are separate choices. Give every tool a narrow schema and deterministic authorization; the model's request is never authorization.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

In [ ]:
TOOL_POLICY = {
    "search_curriculum": {
        "side_effect": False,
        "allowed_groups": {"foundry-learners"},
        "timeout_seconds": 5,
    },
    "publish_release": {
        "side_effect": True,
        "allowed_groups": {"release-owners"},
        "timeout_seconds": 10,
    },
}


def authorize_tool_call(name, caller_groups, approved=False):
    policy = TOOL_POLICY[name]
    if not (policy["allowed_groups"] & set(caller_groups)):
        return False, "caller is not authorized"
    if policy["side_effect"] and not approved:
        return False, "human approval is required"
    return True, "authorized"


assert authorize_tool_call("search_curriculum", {"foundry-learners"})[0]
assert not authorize_tool_call("publish_release", {"release-owners"})[0]
assert authorize_tool_call(
    "publish_release", {"release-owners"}, approved=True
)[0]
TOOL_POLICY

## Agent threat model

Document goal hijacking, tool misuse, identity abuse, MCP supply-chain and data-egress risk, memory poisoning, cascading failure, unbounded consumption, and human over-trust. Add step/token/time budgets, endpoint allow-lists, idempotency keys, a memory retention policy, and an interrupt before side effects.

## Exit criteria

Demonstrate an allowed read, a denied unauthorized call, a blocked side effect awaiting approval, and an idempotent retry. Treat multi-agent orchestration as an advanced core lab; keep A2A-specific work optional when its required components are preview.